# 4 · Values with widths

**The problem.** The logger doesn't send floats. Its radio packet, as the interface control document (ICD) specifies
it, carries the temperature as a signed 16-bit integer in hundredths of a degree, and a status byte whose bit 2 means
"battery low". The firmware engineer reads `int16_t` and `uint8_t`; the rule "the raw temperature is between 200 and
800, and the battery isn't low" is about those types, and about what happens when a value doesn't fit.

Plain ints and floats can't say any of that. Expressions have **value domains**: integers of a width and signedness,
IEEE 754 formats, bits, bytes and more, each named for its standard. This case study writes the packet's rule over
them.

In [1]:
from mbse.Expressions import Domains as D, Evaluators, Expressions as E
from mbse.Schemas.Framework import JSON
from toolkit import show

int16 = D.OfInteger.Builder().width(16).create()
uint8 = D.OfInteger.Builder().width(8).signed(False).create()
int16, uint8

(int16, uint8)

## Step 1: domains are data

A domain is a value, compared by structure: two `int16`s built anywhere are equal. A value of a domain other than a
plain `int` or `float` is a **typed value**, which knows its domain, and refuses a value it can't hold:

In [2]:
raw = D.Value(int16, 523)  # 5.23 °C
status = D.Value(uint8, 0b0000_0100)  # battery low
print(raw, status, D.of(raw) == D.OfInteger.Builder().width(16).create())
try:
    D.Value(uint8, 300)
except Exception as error:
    print(type(error).__name__, error)

523 as int16 4 as uint8 True
ValueError uint8 cannot hold 300


A decoded packet's fields are bound as variables, typed values each. A literal carries its domain too:

In [3]:
packet = {"raw": raw, "status": status}
r, s = E.variable("raw"), E.variable("status")
low, high = E.literal(200, int16), E.literal(800, int16)
in_range = r.ge(low).and_(r.le(high))
print(show(in_range))
Evaluators.OfAny(in_range, packet)

and(ge(raw, int16(200)), le(raw, int16(800)))


True

## Step 2: no implied promotions

C would compare an `int16_t` with the constant `200` by promoting both to `int`. Expressions don't promote: an `int16`
and a plain `int` are different domains, so comparing them is unknown, as `1` and `1.0` are (case study 2):

In [4]:
sloppy = r.ge(200).and_(r.le(800))
print(Evaluators.OfAny(sloppy, packet))

None


A rule that's unknown for every packet is easy to miss. **Inference** catches it before anything runs.
`E.DIALECT.infer(rule, environment)` works out each subexpression's domain from the variables' domains, and refuses an
operation that can't take its arguments:

In [5]:
environment = {"raw": int16, "status": uint8}
print(E.DIALECT.infer(in_range, environment))
try:
    E.DIALECT.infer(sloppy, environment)
except TypeError as error:
    print(error)

bool
ge cannot take (int16, int); it takes (T, T) -> bool for T in int | float | str | bytes or any integer, ieee754, bytes, unicode or packed domain


## Step 3: overflow

An integer of a width has a range, and arithmetic that leaves it **overflows**. What happens then is part of the
domain: `raise` (the default), `wrap`, as unsigned arithmetic does in C, or `saturate`, as a DSP does. The packet's
sequence number is a `uint8` counter that wraps:

In [6]:
counter = D.OfInteger.Builder().width(8).signed(False).overflow("wrap").create()
print(Evaluators.OfAny(E.variable("seq").add(E.literal(1, counter)), {"seq": D.Value(counter, 255)}))
try:
    Evaluators.OfAny(r.add(E.literal(32767, int16)), packet)
except OverflowError as error:
    print(error)

0 as uint8
add overflows int16: 33290


## Step 4: bits

The status byte is flags. `bitand` with a mask isolates bit 2, and a typed literal of the same domain compares the
result; `shr` and `shl` shift:

In [7]:
battery_ok = s.bitand(E.literal(0b100, uint8)).eq(E.literal(0, uint8))
packet_ok = in_range.and_(battery_ok)
print(show(packet_ok))
print(Evaluators.OfAny(packet_ok, packet), Evaluators.OfAny(packet_ok, {**packet, "status": D.Value(uint8, 0)}))

and(and(ge(raw, int16(200)), le(raw, int16(800))), eq(bitand(status, uint8(4)), uint8(0)))
False True


## Step 5: conversions are explicit

The dashboard wants degrees. `convert` keeps a value, changing its domain, and checks it fits; `reinterpret` keeps the
**bits**, as C++'s `bit_cast` does. An operation's `domain` is its result's:

In [8]:
celsius = r.convert(D.Float).mul(0.01)
print(show(celsius), "=", Evaluators.OfAny(celsius, packet))
uint16 = D.OfInteger.Builder().width(16).signed(False).create()
print(Evaluators.OfAny(E.literal(-1, int16).reinterpret(uint16)))
try:
    Evaluators.OfAny(r.convert(uint8), packet)
except OverflowError as error:
    print(error)

mul(convert[float](raw), 0.01) = 5.23
65535 as uint16
convert overflows uint8: 523


## Step 6: domains on the wire

A literal saves with its domain. Registering a domain under a name, once, writes the name instead of its parameters,
which is how an ICD's types become a shared vocabulary:

In [9]:
D.register("int16", int16)
data = low.data
print(JSON.ToJSON(E.Builders).Reachable(E.DIALECT.schema_of(data), data))

{"root": "s0", "objects": {"s0": {"kind": "literal", "int": 200, "domain": {"named": {"name": "int16"}}}}}


## Why it works this way

- **Representation is part of the type.** An ICD describes data words, and a rule about a data word must know its
  width, its signedness and what overflow does. Domains name published standards (two's complement integers, IEEE 754
  formats, VHDL's `std_logic`) so that every language means the same thing by them.
- **No implied promotions.** Each language promotes differently: C to `int`, SystemVerilog to the widest operand. A
  neutral rule promotes nowhere, and a translator writes its language's promotions out (case study 8).
- **Check statically, then run.** Inference finds a rule that can never be decided, before any packet arrives.